# Tracking biofilm movement and breakup
**Time: 1–2 hours. AI assistance is allowed.** Complete the TODOs, check the images,
and explain your results. Save this notebook alongside `biofilm.zip`, `colony_masks.npz`,
and `breakup_scores.csv`. Requirements: numpy, pandas, tifffile, matplotlib, Pillow.

The masks already preserve colony IDs. Your job is to measure territory-centroid movement
and analyze the supplied teaching breakup scores. Scores range from 0 to 100; higher indicates
greater fragmentation according to the supplied indicator. They are not percentages of released
cells or biomass. Their generation is outside this exercise.

These are selected annotations on Chou et al. (2026), Supplementary Video 1.
Not every visible colony has a supplied mask. Use the listed colony IDs only.
Measurements use pixels and frame numbers because physical calibration is unverified.
Centroid movement can also reflect changing territory shape.

Submit ONLY this completed, saved notebook (.ipynb), with its outputs retained. Include both
results tables, the mask overlay, trajectory animation, movement/breakup figure, and a 100–150 word
interpretation. The final cell embeds your tables for grading. No separate CSVs or GIF are required.
Automatic measurements: 70 points.
Loading/alignment/completeness: 10 points. Visualization: 10 points. Interpretation: 10 points.


In [ ]:
from pathlib import Path
import io, zipfile
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Image as NotebookImage
DATA = Path.cwd()
OUTPUT = DATA / 'submission'
OUTPUT.mkdir(exist_ok=True)
with zipfile.ZipFile(DATA / 'biofilm.zip') as archive:
    names = sorted(archive.namelist())
print('Images:', names[:3], '...')


## 1. Load and display an image
The first image is loaded for you. Complete the second example. TIFFs are 2D arrays:
the first index is row (y), the second is column (x).


In [ ]:
with zipfile.ZipFile(DATA / 'biofilm.zip') as archive:
    first_image = tifffile.imread(io.BytesIO(archive.read(names[0])))
plt.imshow(first_image, cmap='gray'); plt.title('First image'); plt.show()

# TODO: choose frame 1 or later, load it, and display it.
frame_to_show = 1
another_image = None  # replace with your image-loading code


## 2. Load and overlay the masks
Image–mask pairs have already been matched by the organizers. `image_filenames` gives
their exact order. Load the mask array, then overlay the mask for `frame_to_show`.
Confirm that boundaries follow the matching raw image. Unknown calibration fields contain NaN.


In [ ]:
with np.load(DATA / 'colony_masks.npz', allow_pickle=False) as archive:
    print('Available fields:', archive.files)
    filenames = archive['image_filenames'].copy()
    colony_ids = archive['colony_ids'].copy()
    masks = None  # TODO: load the 'masks' array
assert masks is not None, 'Load masks before continuing'
with zipfile.ZipFile(DATA / 'biofilm.zip') as archive:
    images = np.stack([tifffile.imread(io.BytesIO(archive.read('biofilm/' + str(name)))) for name in filenames])
assert images.shape == masks.shape
# TODO: display images[frame_to_show] and overlay masks[frame_to_show].
# Hint: use a masked array to make mask value 0 transparent.


## 3. Measure a colony centroid
Here is a complete example. x is column, y is row. Use the territory mask, not brightness.
Repeat for a second colony and draw its centroid on the image.


In [ ]:
example_id = int(colony_ids[0])
yy, xx = np.nonzero(masks[0] == example_id)
example_x, example_y = xx.mean(), yy.mean()
print(example_id, example_x, example_y)
plt.imshow(images[0], cmap='gray')
plt.plot(example_x, example_y, 'r+')
plt.show()
# TODO: repeat for another colony.


## 4. Follow every colony
Complete `measure_centroids`. Return one row for every colony and every frame, including
absent observations as NaN coordinates. Columns: `colony_id, frame, centroid_x_px, centroid_y_px`.
Hint: start from the example above and add loops. Do not replace missing coordinates with zero.


In [ ]:
def measure_centroids(masks, colony_ids):
    rows = []
    # TODO: loop over colony IDs and frames; find matching pixels and record their mean x/y.
    raise NotImplementedError('Complete centroid measurements')

centroids = measure_centroids(masks, colony_ids)
assert len(centroids) == len(masks) * len(colony_ids)
example = centroids[(centroids.colony_id == example_id) & (centroids.frame == 0)].iloc[0]
assert np.isclose(example.centroid_x_px, example_x)
assert np.isclose(example.centroid_y_px, example_y)
centroids.head()


## 5. Calculate movement
Displacement is Euclidean distance from the first observed centroid. Step distance uses
consecutive observed frames only. Cumulative distance sums valid steps. The first observed
frame has step distance 0 if it is frame 0; otherwise its step is NaN. On absent frames,
all movement entries are NaN. After a gap, resume the existing cumulative total without
adding a step across the gap.

Example: centroids (0,0), (3,4), (3,0) give displacements 0,5,3 and cumulative distances 0,5,9.
Return the original columns plus `displacement_px, step_distance_px, cumulative_distance_px`.


In [ ]:
def calculate_movement(centroids):
    # TODO: group by colony, sort by frame, and calculate the three distances.
    raise NotImplementedError('Complete movement calculations')

movement = calculate_movement(centroids)
movement.head()


## 6. Analyze supplied breakup scores
Load the CSV using the example below. For each colony, calculate `breakup_change` relative to
its first finite score. Identify the first frame with score >= THRESHOLD, including frame 0.
No crossing stays NaN. Scores can decrease. Use the supplied rounded scores exactly.
Return the scores with a `breakup_change` column and a table of `colony_id, first_threshold_frame`.


In [ ]:
THRESHOLD = 40.0
scores = pd.read_csv(DATA / 'breakup_scores.csv')
display(scores.head())

def analyze_breakup(scores, threshold):
    # TODO: calculate changes and first crossings for every colony.
    raise NotImplementedError('Complete breakup analysis')

scored_frames, crossings = analyze_breakup(scores, THRESHOLD)


## 7. Combine the measurements and summarize colonies
Merge movement and breakup results on BOTH `colony_id` and `frame`. Frame results need:
`colony_id, frame, centroid_x_px, centroid_y_px, displacement_px, step_distance_px,
cumulative_distance_px, breakup_score, breakup_change`.

Summary results need one row per colony: `colony_id, final_displacement_px, total_distance_px,
breakup_change, first_threshold_frame`. Use the last observed centroid for final displacement,
the last finite cumulative distance, and last finite score minus first finite score.
Which colony moves farthest from its starting point? Which score increases most?


In [ ]:
# TODO: construct these two tables using your results.
measurements = None
summary = None


## 8. Visualize
The helper draws your measurements on raw frames, colored by supplied breakup score.
Pass your results into it and display the animation using `display_trajectory_gif` below.
That helper embeds the image bytes into the notebook so they remain visible after submission.
Also create a two-panel figure for one colony: displacement and cumulative distance above,
breakup score with the threshold and first crossing marked below. Share the frame axis.
Display the figure in the notebook. Trails break at missing observations.


In [ ]:
def save_trajectory_gif(images, measurements, output_path):
    frames = []
    for frame in range(len(images)):
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.imshow(images[frame], cmap='gray')
        current = measurements[measurements.frame == frame]
        for ident, track in measurements.groupby('colony_id'):
            trail = track[track.frame <= frame].sort_values('frame')
            ax.plot(trail.centroid_x_px, trail.centroid_y_px, color='cyan', linewidth=0.8, alpha=0.65)
        valid = current.dropna(subset=['centroid_x_px', 'centroid_y_px'])
        points = ax.scatter(valid.centroid_x_px, valid.centroid_y_px, c=valid.breakup_score, cmap='viridis', vmin=0, vmax=100, s=25)
        for _, row in valid.iterrows():
            ax.text(row.centroid_x_px+3, row.centroid_y_px, str(int(row.colony_id)), color='white', fontsize=8)
        fig.colorbar(points, ax=ax, label='Supplied breakup score')
        ax.set(title=f'Frame {frame}', xlabel='x (pixels)', ylabel='y (pixels)')
        fig.tight_layout()
        buffer = io.BytesIO(); fig.savefig(buffer, format='png', dpi=85); plt.close(fig)
        buffer.seek(0); frames.append(Image.open(buffer).convert('RGB'))
    frames[0].save(output_path, save_all=True, append_images=frames[1:], duration=300, loop=0)

def display_trajectory_gif(images, measurements):
    local_gif = OUTPUT / 'trajectories.gif'
    save_trajectory_gif(images, measurements, local_gif)
    display(NotebookImage(data=local_gif.read_bytes(), format='gif'))

# TODO: call display_trajectory_gif(images, measurements).
# TODO: display the two-panel movement/breakup figure for a colony you choose.


## 9. Store results in this notebook and check
Numerical grading uses absolute tolerances: 0.05 pixels for centroids, 0.10 pixels for
distances, 0.01 score points for score changes, and exact frame numbers for crossings.
Partial credit is the fraction of correct required entries; missing or nonfinite answers
fail finite reference entries. Duplicate colony/frame rows are invalid. Keep at least 3 decimals
in your calculations. Use NaN for missing observations.

70 automatic points: centroids 20, movement 30, breakup changes/crossings 20.
30 review points: loading/overlay/complete outputs 10, visualization 10, interpretation 10.


In [ ]:
assert measurements is not None and summary is not None
assert not measurements.duplicated(['colony_id', 'frame']).any()
assert not summary.duplicated(['colony_id']).any()
expected = pd.MultiIndex.from_product([colony_ids, range(len(masks))], names=['colony_id', 'frame'])
actual = pd.MultiIndex.from_frame(measurements[['colony_id', 'frame']])
assert set(actual) == set(expected), 'Include every required colony/frame pair'
assert set(summary.colony_id) == set(colony_ids)
# Show complete tables in expandable sections, with full numeric results embedded for grading.
import json
payload = {'biofilm_challenge_version': 1,
    'measurements': json.loads(measurements.to_json(orient='records', double_precision=12)),
    'summary': json.loads(summary.to_json(orient='records', double_precision=12))}
html = ('<details><summary>Frame-by-frame measurements</summary>' + measurements.to_html(index=False) + '</details>'
        + '<details><summary>Colony summaries</summary>' + summary.to_html(index=False) + '</details>')
display({'application/json': payload, 'text/html': html,
         'text/plain': 'Complete challenge results: frame measurements and colony summaries'}, raw=True)
print('Results are now embedded. Save this notebook with outputs retained before submitting.')


## 10. Your interpretation (100–150 words)
Name the colony with greatest final displacement and the colony with greatest score increase.
Discuss one example using your graph and images. Explain one limitation: for example,
centroid shifts can reflect shape changes, or a supplied breakup score does not prove cell release.
Write your response here. Confirm the overlay, animation, two-panel figure, and both tables are
visible above, then SAVE the notebook. Submit that one .ipynb file. Do not clear its outputs.
The animation must be embedded, rather than linked to a file on your computer.
